# Pipeline EDA — Opción B: comercio electrónico

Pipeline de 10 etapas sobre *eCommerce behavior data (multi-category store)* + componente sintético.
La lógica vive en `pipeline_eda/`; este notebook solo importa, ejecuta y muestra resultados.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Busca hacia arriba la carpeta que contiene pipeline_eda/, sin depender del cwd del kernel.
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pipeline_eda").is_dir())
sys.path.insert(0, str(RAIZ))

In [2]:
from pipeline_eda.etapa0_ingesta import TAMANOS, leer_csv
from pipeline_eda.orquestador import Config, Pipeline

In [3]:
from pipeline_eda.etapa0_ingesta import leer_csv

eventos = leer_csv(RAIZ / "data" / "2019-Oct-10M.csv")

In [4]:
next(eventos)

Event(event_time=datetime.datetime(2019, 10, 1, 0, 0, tzinfo=datetime.timezone.utc), event_type='view', product_id=44600062, category_id=2103807459595387724, category_code=None, brand='shiseido', price=35.79, user_id=541312140, user_session='72d76fde-8bb3-4e00-8c23-a032dfed738c', order_id=None, tipo_envio=None, card_id=None, is_fraud=False, is_dup=False)

## 1. Datos

In [5]:
TAMANO = "10M"  # uno de TAMANOS: 1K, 10K, 100K, 1M, 10M
# Eventos reales; la etapa 0 agrega el componente sintético (con SEMILLA).
fuente = leer_csv(RAIZ / "data" / "2019-Oct-10M.csv", limite=TAMANOS[TAMANO])

## 2. Ejecución de punta a punta (etapas 0 → 9)

In [6]:
config = Config(seed=42)
pipe = Pipeline(config)
pipe.run(fuente)

## 3. Métricas por etapa

In [7]:
pipe.metricas()

{'etapa0_ingesta': {'ingeridos': 10204051,
  'lotes': 1021,
  'ocupacion_max': 10000,
  'bloqueos': 0,
  'sintetico': {'duplicados_reales': 6213,
   'duplicados_inyectados': 200391,
   'rafagas_fraude': 366,
   'intentos_fraude': 3660,
   'tarjetas_senaladas': 50}},
 'etapa1_preproceso': {'recibidos': 10204051,
  'invalidos': 15822,
  'invalidos_por_motivo': {'precio': 15821, 'sin_sesion': 1},
  'duplicados_lote': 48865,
  'duplicados_ventana': 156485,
  'emitidos': 9982879,
  'tamano_ventana': 100000},
 'etapa2_prioridad': {'encolados_por_prioridad': {'ALERTA': 4376,
   'EXPRESS': 35831,
   'ESTANDAR': 105192,
   'PROGRAMADA': 35473,
   'NAVEGACION': 9806383},
  'tamano_max': 9946},
 'etapa3_cache': {'productos': 120528,
  'tasa_aciertos_lru': None,
  'celdas_no_nulas': 5798912},
 'etapa4_bplustree': {'eventos_procesados': 9979974,
  'productos_indexados': 120528,
  'arbol_precio': {'claves_indexadas': 36121,
   'altura': 3,
   'consultas': 0,
   'paginas_leidas_ultima_consulta': 0,
 

## 4. Prioridades y fraude (etapas 2 y 6)

In [8]:
pipe.e9.alertas[:10]

[(Alerta(tipo='senalada', clave='T514892797', conteo_estimado=1, evento=Event(event_time=datetime.datetime(2019, 10, 1, 2, 32, 14, tzinfo=datetime.timezone.utc), event_type='purchase', product_id=1004446, category_id=2053013555631882655, category_code='electronics.smartphone', brand='samsung', price=89.32, user_id=514892797, user_session='c88267b8-c2b0-478a-9d98-9d60ebde43b7', order_id='c88267b8-c2b0-478a-9d98-9d60ebde43b7', tipo_envio='express', card_id='T514892797', is_fraud=False, is_dup=False)),
  False),
 (Alerta(tipo='senalada', clave='T555465061', conteo_estimado=1, evento=Event(event_time=datetime.datetime(2019, 10, 1, 2, 49, 44, tzinfo=datetime.timezone.utc), event_type='purchase', product_id=2800403, category_id=2053013563835941749, category_code='appliances.kitchen.refrigerators', brand=None, price=204.33, user_id=555465061, user_session='87515b74-457c-4272-8fcf-5739f0196b4e', order_id='87515b74-457c-4272-8fcf-5739f0196b4e', tipo_envio='express', card_id='T555465061', is_fra

## 5. Consultas de productos (etapas 3 y 4)

In [9]:
pipe.e3.obtener_producto(1004237)
pipe.e4.productos_en_rango_precio(100.0, 200.0)[:10]

[1001588,
 1002442,
 1002482,
 1002484,
 1002757,
 1002876,
 1002974,
 1003001,
 1003013,
 1003023]

## 6. Productos relacionados (etapas 5 y 7)

In [10]:
pid = 1004237
pipe.e5.similares(pid, k=10), pipe.e7.relacionados(pid, k=10), pipe.e7.similares_vectoriales(pid, k=10)

([1004234,
  1005116,
  1005119,
  1005122,
  1004201,
  1003318,
  1003319,
  1004240,
  1004484,
  1004229],
 [],
 [1004234,
  1005116,
  1005119,
  1005122,
  1004201,
  1003318,
  1003319,
  1004240,
  1004484,
  1004229])

## 7. Grafo de co-compra (etapa 8)

In [11]:
pr = pipe.e8.pagerank()
comunidades = pipe.e8.comunidades()
sorted(pr.items(), key=lambda x: -x[1])[:10], pipe.e8.modularidad(comunidades)

([(1004856, 0.014522872929149513),
  (1004767, 0.011236521735884738),
  (1004833, 0.007138198335905187),
  (1005115, 0.006983270715457772),
  (1002544, 0.006662538325408957),
  (1004870, 0.005341006207077954),
  (1002524, 0.0047055734179650525),
  (1004836, 0.004349785404614137),
  (1004873, 0.004160317366117688),
  (1004249, 0.004064406946604291)],
 0.3571290362425227)

## 8. Servicio (etapa 9)

In [12]:
pipe.e9.autocompletar("electronics.smart", k=5)

['electronics.smartphone']

In [13]:
pipe.e9.populares(k=10), pipe.e9.recomendar(pid, k=10)

([(1004856, 6787),
  (1004767, 4889),
  (1005115, 3064),
  (4804056, 3027),
  (1004833, 2782),
  (1004870, 2610),
  (1002544, 2497),
  (1004249, 2021),
  (5100816, 1838),
  (1002524, 1721)],
 [1005105,
  1002544,
  1003317,
  1004246,
  1005115,
  1005132,
  1003316,
  1004227,
  1003310,
  1005104])

In [14]:
pipe.e9.dashboard(pipe.metricas())

=== Ventas ===
unidades 173,591  ingresos 54,007,709.00  productos vendidos 19,214
más vendidos:
  1004856 samsung · electronics.smartphone · 131.02  (6787 unidades)
  1004767 samsung · electronics.smartphone · 250.96  (4889 unidades)
  1005115 apple · electronics.smartphone · 1015.46  (3064 unidades)
  4804056 apple · electronics.audio.headphone · 160.24  (3027 unidades)
  1004833 samsung · electronics.smartphone · 170.92  (2782 unidades)
más influyentes (PageRank): 1004856, 1004767, 1004833, 1005115, 1002544
=== Actividad ===
cart 193,709  purchase 173,591  view 9,612,674
=== Alertas ===
emitidas 4,376  bloqueadas 2,905 {'intensidad': 1095, 'senalada': 1810}  liberadas (falso positivo) 1,471
=== Métricas por etapa ===
etapa0_ingesta: {'ingeridos': 10204051, 'lotes': 1021, 'ocupacion_max': 10000, 'bloqueos': 0, 'sintetico': {'duplicados_reales': 6213, 'duplicados_inyectados': 200391, 'rafagas_fraude': 366, 'intentos_fraude': 3660, 'tarjetas_senaladas': 50}}
etapa1_preproceso: {'recibi